# **Figure 1**. Temporal evolution of community-level metrics: (a,d,g,j) normalized largest-community size $S_1(t)/N(t)$, (b,e,h,k) number of detected communities $n_s(t)$, and (c,f,i,l) normalized active network size $N(t)/N$, for networks with $\langle k \rangle = 9, 18, 36,$ and $54$ from top to bottom. Blue and red curves correspond to $\varepsilon = 0.30$ and $\varepsilon = 0.50$, respectively.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Keep manuscript LaTeX style
USE_TEX = True
ALLOW_TEX_FALLBACK = False  # set True only if you want automatic non-TeX fallback
# Epsilon glyph in legends/labels
EPS_SYMBOL = r"\varepsilon"
plt.rc('text', usetex=USE_TEX)
plt.rc('font', family='serif')

# Probe TeX once so we fail early (or fallback, if enabled)
try:
    _probe = plt.figure(figsize=(1, 1))
    _probe.text(0.5, 0.5, rf'${EPS_SYMBOL}$')
    _probe.canvas.draw()
    plt.close(_probe)
except Exception as exc:
    plt.close('all')
    if ALLOW_TEX_FALLBACK:
        USE_TEX = False
        plt.rc('text', usetex=False)
        print(f'[warn] TeX unavailable ({exc.__class__.__name__}); using mathtext fallback.')
    else:
        raise RuntimeError(
            'LaTeX rendering is required for this figure, but TeX is unavailable in this environment. '
            'Install/configure TeX (or set ALLOW_TEX_FALLBACK=True).'
        ) from exc

# -------------------------
# Config
# -------------------------
P_TO_K = {"0.030": 9, "0.060": 18, "0.120": 36, "0.180": 54}
K_ORDER = [9, 18, 36, 54]
EPSILONS = [0.30, 0.50]
RESULTS_ROOT = Path("results")
OUTPUT_DIR = Path("plots_paper/fig1")
OUTPUT_BASENAME = "fig1"


# Typography (scaled up for LaTeX readability)
FS_TICK = 12
FS_LABEL = 14
FS_TITLE = 16
FS_PANEL = 13
FS_ROWLABEL = 14
FS_LEGEND = 14

# Keep coherence with your previous color convention
fallback_color_map = {
    "0.300": "b",
    "0.500": "r",
}


def _clean_columns_local(df):
    if "_clean_columns" in globals():
        return _clean_columns(df)

    cleaned = {}
    for c in df.columns:
        c2 = c.strip().replace("<", "").replace(">", "").replace(" ", "")
        cleaned[c] = c2
    return df.rename(columns=cleaned)


def _std_or_zero_local(series):
    v = series.std(ddof=1)
    return 0.0 if pd.isna(v) else float(v)


def _pad_limits_local(y_min, y_max, pad_ratio=0.05):
    if "_pad_limits" in globals():
        return _pad_limits(y_min, y_max, pad_ratio=pad_ratio)

    if not np.isfinite(y_min) or not np.isfinite(y_max):
        return (0.0, 1.0)
    if y_min == y_max:
        delta = max(abs(y_min) * 0.05, 1e-3)
        return (y_min - delta, y_max + delta)
    span = y_max - y_min
    pad = span * pad_ratio
    return (y_min - pad, y_max + pad)


def _aggregate_metrics(df):
    df = df.copy()
    N0 = float(df["N"].max())
    df["s1_over_N"] = df["s1"] / df["N"]
    df["N_over_N0"] = df["N"] / N0
    metrics = ["s1_over_N", "n_s", "N", "N_over_N0"]

    agg = (
        df.groupby(["epsilon", "time_step"])[metrics]
        .agg(["mean", _std_or_zero_local])
        .rename(columns={"_std_or_zero_local": "std"})
    )
    agg.columns = [f"{m}_{s}" for m, s in agg.columns]
    return agg.reset_index()


def _eps_color(eps, idx=0):
    eps_key = f"{eps:.3f}"
    if "color_map" in globals() and isinstance(color_map, dict):
        if eps_key in color_map:
            return color_map[eps_key]
    return fallback_color_map.get(eps_key, plt.cm.tab10(idx % 10))


def _label_eps(eps):
    return fr"${EPS_SYMBOL}={eps:.2f}$"


def _condition_stats(agg, eps):
    d = agg[agg["epsilon"] == eps].sort_values("time_step")
    if d.empty:
        return None

    t = d["time_step"].to_numpy()
    s1n = d["s1_over_N_mean"].to_numpy()
    ns = d["n_s_mean"].to_numpy()
    n = d["N_mean"].to_numpy()

    i_ns_peak = int(np.argmax(ns))
    i_n_min = int(np.argmin(n))

    n_start = float(n[0])
    n_min = float(n[i_n_min])
    n_final = float(n[-1])

    return {
        "t": t,
        "s1n_start": float(s1n[0]),
        "s1n_final": float(s1n[-1]),
        "s1n_drop": float(s1n[0] - s1n[-1]),
        "ns_peak": float(ns[i_ns_peak]),
        "ns_peak_t": float(t[i_ns_peak]),
        "N_start": n_start,
        "N_min": n_min,
        "N_final": n_final,
        "N_drop_abs": float(n_start - n_min),
        "N_drop_rel": float((n_start - n_min) / n_start) if n_start > 0 else np.nan,
    }


# -------------------------
# Load and aggregate
# -------------------------
metric_cfg = [
    ("s1_over_N", r"$S_1(t)/N(t)$", r"$S_1(t)/N(t)$ vs time"),
    ("n_s", r"$n_s(t)$", r"$n_s(t)$ vs time"),
    ("N_over_N0", r"$N(t)/N$", r"$N(t)/N$ vs time"),
]

by_k = {}
global_x_min = np.inf
global_x_max = -np.inf
global_y = {m: [np.inf, -np.inf] for m, _, _ in metric_cfg}

for p_str, k_val in P_TO_K.items():
    path = RESULTS_ROOT / f"results_p{p_str}" / "network_metrics.txt"
    if not path.exists():
        print(f"[warn] Missing file: {path}")
        continue

    df = pd.read_csv(path)
    df = _clean_columns_local(df)

    required = ["epsilon", "simulation", "time_step", "s1", "n_s", "N"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[warn] Missing columns for k={k_val}: {missing}")
        continue

    for c in required:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=required).copy()

    df = df[df["epsilon"].isin(EPSILONS)].copy()
    if df.empty:
        print(f"[warn] No rows for selected epsilons in k={k_val}")
        continue

    agg = _aggregate_metrics(df)
    by_k[k_val] = {"agg": agg, "path": path}

    global_x_min = min(global_x_min, float(agg["time_step"].min()))
    global_x_max = max(global_x_max, float(agg["time_step"].max()))

    for m, _, _ in metric_cfg:
        lo = (agg[f"{m}_mean"] - agg[f"{m}_std"]).min()
        hi = (agg[f"{m}_mean"] + agg[f"{m}_std"]).max()
        global_y[m][0] = min(global_y[m][0], float(lo))
        global_y[m][1] = max(global_y[m][1], float(hi))

if not by_k:
    raise RuntimeError("No valid datasets found for the compact community diagnostics section.")

missing_rows = [k for k in K_ORDER if k not in by_k]
if missing_rows:
    print(f"[warn] Missing k rows (no valid data): {missing_rows}")

global_y_padded = {m: _pad_limits_local(v[0], v[1], pad_ratio=0.05) for m, v in global_y.items()}


# -------------------------
# Plot compact figure
# -------------------------
plt.style.use("seaborn-v0_8-whitegrid")
# Re-apply text/font settings after style load
plt.rc('text', usetex=USE_TEX)
plt.rc('font', family='serif')
fig, axs = plt.subplots(len(K_ORDER), len(metric_cfg), figsize=(17, 16.5), sharex=True, squeeze=False)
panel_labels = list("abcdefghijkl")
label_idx = 0

for r, k_val in enumerate(K_ORDER):
    for c, (m, ylab, col_title) in enumerate(metric_cfg):
        ax = axs[r, c]

        if k_val not in by_k:
            ax.axis("off")
            label_idx += 1
            continue

        agg = by_k[k_val]["agg"]

        for eidx, eps in enumerate(EPSILONS):
            d = agg[agg["epsilon"] == eps].sort_values("time_step")
            if d.empty:
                continue

            t = d["time_step"].to_numpy()
            mean = d[f"{m}_mean"].to_numpy()
            std = d[f"{m}_std"].to_numpy()
            color = _eps_color(eps, idx=eidx)

            ax.plot(t, mean, lw=2.2, color=color, label=_label_eps(eps), zorder=3)
            ax.fill_between(t, mean - std, mean + std, color=color, alpha=0.22, zorder=2)

        ax.set_xlim(global_x_min, global_x_max)
        if m == "N_over_N0":
            ax.set_ylim(0.0, 1.1)
        else:
            ax.set_ylim(*global_y_padded[m])
        ax.grid(True, alpha=0.35)
        ax.set_axisbelow(True)
        ax.tick_params(axis="both", labelsize=FS_TICK)

        panel_tag = f"({panel_labels[label_idx]})"
        if c == 2:
            # Move right-column panel labels away from the near-flat N curves
            ax.text(
                0.02,
                0.05,
                panel_tag,
                transform=ax.transAxes,
                va="bottom",
                ha="left",
                fontsize=FS_PANEL,
                fontweight="bold",
            )
        else:
            ax.text(
                0.02,
                0.96,
                panel_tag,
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=FS_PANEL,
                fontweight="bold",
            )
        label_idx += 1

        if r == 0:
            ax.set_title(col_title, fontsize=FS_TITLE)

        if c == 0:
            ax.set_ylabel(ylab, fontsize=FS_LABEL)
            ax.text(
                -0.22,
                0.5,
                fr"$\langle k \rangle = {k_val}$",
                transform=ax.transAxes,
                rotation=90,
                va="center",
                ha="center",
                fontsize=FS_ROWLABEL,
            )
        else:
            ax.set_ylabel(ylab, fontsize=FS_LABEL)

        if r == len(K_ORDER) - 1:
            ax.set_xlabel("Adaptive time steps", fontsize=FS_LABEL)
            ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)

handles = [
    plt.Line2D([0], [0], color=_eps_color(eps, idx=i), lw=3, label=_label_eps(eps))
    for i, eps in enumerate(EPSILONS)
]
fig.legend(
    handles=handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.03),
    ncol=2,
    fontsize=FS_LEGEND,
    frameon=False,
)

fig.tight_layout(rect=[0.03, 0.03, 1, 0.93])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / f"{OUTPUT_BASENAME}.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / f"{OUTPUT_BASENAME}.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Final figure saved as plots_paper/fig1/fig1.[pdf|png]")


# **Figure 2**. (a-c) Average clustering coefficient $\langle C \rangle$ and temporal evolution of (d-f) global clustering coefficient $C$, (g-i) small-world index $\sigma$, and (j-l) average shortest path length $L$.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# === Style settings ===
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

fontsize_labels = 36
fontsize_ticks = 28
fontsize_titles = 38
fontsize_panel_labels = 34
fontsize_legend = 34
panel_spacing = 0.25
point_size_scatter = 150

# === Global simulation selector (shared across all figures) ===
SIMULATION_ID = 1
SIMULATION_IDS = [SIMULATION_ID]

# === Color map ===
color_map = {
    "0.100": 'tab:orange', "0.150": 'tab:cyan', "0.200": 'tab:purple', "0.250": 'tab:brown',
    "0.300": 'b', "0.350": 'tab:pink', "0.360": 'tab:olive', "0.370": 'tab:gray',
    "0.400": 'g', "0.450": 'tab:blue', "0.500": 'r'
}

# === Mapping from p to <k> ===
p_to_k = {
    "0.060": 18,
    "0.120": 36,
    "0.180": 54
}

# === Metrics and axis limits ===
dynamic_metrics = ["clustering", "omega", "path_length"]
metric_labels = [r"$\langle C \rangle$", r"$C$", r"$\sigma$", r"$L$"]
y_limits = [
    (0.0, 0.9),   # <C>
    (0.0, 0.9),   # C
    (0.0, 10.0),  # sigma, stored as omega in network_metrics.txt
    (1.7, 3.2)    # L
]

# === Initialize figure ===
fig, axs = plt.subplots(3, 4, figsize=(30, 20))
panel_labels = [f"({chr(97+i)})" for i in range(12)]
panel_label_matrix = np.array(panel_labels).reshape(4, 3).T

# === Get epsilon values and legend once from the first file
first_folder = next(iter(p_to_k))
df_temp = pd.read_csv(os.path.join(f"results/results_p{first_folder}", "network_metrics.txt"))
epsilons_for_legend = sorted(df_temp["epsilon"].unique())
legend_handles = [
    plt.Line2D([0], [0], color=color_map[f"{eps:.3f}"], lw=2.8, label=rf"$\varepsilon={eps:.2f}$")
    for eps in epsilons_for_legend
]

# === Loop through each p folder ===
for row_idx, (p_str, k_val) in enumerate(p_to_k.items()):
    folder = f"results/results_p{p_str}"
    path = os.path.join(folder, "network_metrics.txt")
    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)
    df.rename(columns={"<s>": "s_mean", "<s>_std": "s_std"}, inplace=True)
    epsilon_values = sorted(df["epsilon"].unique())
    n_last = 100000

    # Panel for <C>
    ax = axs[row_idx, 0]
    ax.set_ylim(y_limits[0])
    avg_clustering = []
    std_clustering = []

    for eps in epsilon_values:
        df_eps = df[df["epsilon"] == eps]
        max_step = df_eps["time_step"].max()
        df_final = df_eps[df_eps["time_step"] >= max_step - n_last]
        grouped = df_final.groupby("simulation")["clustering"].mean()
        avg_clustering.append(grouped.mean())
        std_clustering.append(grouped.std())

    eps_array = np.array(epsilon_values)
    colors = [color_map.get(f"{eps:.3f}", 'k') for eps in eps_array]
    ax.scatter(eps_array, avg_clustering, c=colors, s=point_size_scatter, zorder=2)
    ax.plot(eps_array, avg_clustering, color='black', linewidth=1.2, zorder=1)
    ax.errorbar(eps_array, avg_clustering, yerr=std_clustering,
                fmt='none', ecolor='black', elinewidth=2.0, capsize=3, capthick=1.5, zorder=3)

    if row_idx == 2:
        ax.set_xlabel(r"$\varepsilon$", fontsize=fontsize_labels)
    if row_idx == 0:
        ax.set_title(metric_labels[0], fontsize=fontsize_titles)

    ax.set_ylabel(rf"$\langle k \rangle = {k_val}$", fontsize=fontsize_labels)
    ax.grid(True)
    ax.tick_params(axis='both', labelsize=fontsize_ticks)
    ax.tick_params(axis='x', labelrotation=0)
    for tick in ax.get_xticklabels():
        tick.set_horizontalalignment('right')
    ax.text(0.02, 0.93, panel_label_matrix[row_idx, 0], transform=ax.transAxes,
            fontsize=fontsize_panel_labels, fontweight='bold', va='top', ha='left')

    # Panels for C, sigma, L
    for col_offset, (metric, label, y_lim) in enumerate(zip(dynamic_metrics, metric_labels[1:], y_limits[1:]), start=1):
        ax = axs[row_idx, col_offset]
        ax.set_ylim(y_lim)

        for eps in epsilon_values:
            df_eps = df[df["epsilon"] == eps]
            grouped = df_eps.groupby("time_step")[metric].agg(['mean', 'std']).reset_index()
            eps_str = f"{eps:.3f}"
            color = color_map.get(eps_str, 'k')

            ax.plot(grouped["time_step"], grouped["mean"], color=color, label=rf"$\varepsilon={eps:.2f}$")
            ax.fill_between(grouped["time_step"],
                            grouped["mean"] - grouped["std"],
                            grouped["mean"] + grouped["std"],
                            color=color, alpha=0.2)

        if row_idx == 2:
            ax.set_xlabel("Adaptive time steps", fontsize=fontsize_labels)
        if row_idx == 0:
            ax.set_title(label, fontsize=fontsize_titles)

        ax.grid()
        ax.tick_params(axis='both', labelsize=fontsize_ticks)
        ax.tick_params(axis='x', labelrotation=0)
        for tick in ax.get_xticklabels():
            tick.set_horizontalalignment('right')
        ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
        ax.ticklabel_format(style='sci', axis='x', scilimits=(0, 0))
        ax.xaxis.get_offset_text().set_fontsize(fontsize_ticks + 2)

        ax.text(0.02, 0.93, panel_label_matrix[row_idx, col_offset], transform=ax.transAxes,
                fontsize=fontsize_panel_labels, fontweight='bold', va='top', ha='left')

# === Global legend (bottom center)
legend = fig.legend(
    handles=legend_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.015),
    ncol=4,
    fontsize=fontsize_legend,
    frameon=True,
    handlelength=2.8,
    columnspacing=1.8,
    labelspacing=1.0,
    borderpad=0.7,
    handletextpad=0.8,
    alignment='center'
)
legend.get_frame().set_edgecolor('black')

# === Save output
output_dir = "plots_paper/fig2"
os.makedirs(output_dir, exist_ok=True)
plt.tight_layout(rect=[0, 0.08, 1, 1])
fig.subplots_adjust(bottom=0.10, wspace=panel_spacing, hspace=panel_spacing)
plt.savefig(os.path.join(output_dir, "fig2.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(output_dir, "fig2.png"), dpi=300, bbox_inches="tight")
plt.close()

print("Final figure saved as plots_paper/fig2/fig2.[pdf|png]")


# **Figure 3**. Degree Distribution.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# === Plot style settings ===
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

SIMULATION_ID = globals().get("SIMULATION_ID", 1)
SIMULATION_IDS = globals().get("SIMULATION_IDS", [SIMULATION_ID])

# === General configuration ===
k_to_p = {18: 0.060, 36: 0.120, 54: 0.180}
epsilons = [0.3, 0.5]
steps = [0, 10_000, 100_000, 300_000]
step_titles = [
    r"Adaptive time step $0$",
    r"Adaptive time step $1 \times 10^{4}$",
    r"Adaptive time step $1 \times 10^{5}$",
    r"Adaptive time step $3 \times 10^{5}$"
]

# === Customizable parameters ===
N = 300  # total number of nodes (constant)
max_grado = N - 1
ylim_y = 0.12  # Expected maximum height for P(k)
xlim_x = 180
mostrar_dispersion = False  # Set to True to show +/-std shaded areas

# === Visual style ===
BIN_WIDTH = 4  # one bin for each integer degree (k=0,1,2,...)
bin_edges = np.arange(0, max_grado + BIN_WIDTH + 1, BIN_WIDTH)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_widths = np.diff(bin_edges)
color_map = {0.3: 'blue', 0.5: 'red'}
label_map = {0.3: r"$\varepsilon = 0.3$", 0.5: r"$\varepsilon = 0.5$"}
alfa_mean = 0.6
alfa_std = 0.25

# === File path templates ===
RESULTS_ROOT_CANDIDATES = ["results", "results_paper", "."]
output_dir = "plots_paper/fig3"
os.makedirs(output_dir, exist_ok=True)


def _candidate_matrix_paths(p_val, sim_id, eps, step):
    eps_tag = f"{eps:.3f}"
    rel_path = os.path.join(
        f"results_p{p_val:.3f}",
        f"simulation_{sim_id}",
        "networks",
        f"eps_{eps_tag}",
        f"matrix_eps_{eps_tag}_step_{step}.txt",
    )
    candidates = []
    for root in RESULTS_ROOT_CANDIDATES:
        if root == ".":
            candidates.append(rel_path)
        else:
            candidates.append(os.path.join(root, rel_path))
    return candidates

# === Loop over different k values ===
for k, p in k_to_p.items():
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()

    for idx, (step, title) in enumerate(zip(steps, step_titles)):
        ax = axs[idx]

        for eps in epsilons:
            all_probs = []

            for sim_id in SIMULATION_IDS:
                candidates = _candidate_matrix_paths(p, sim_id, eps, step)
                path = next((pp for pp in candidates if os.path.exists(pp)), None)
                if path is None:
                    print(f"Not found: {' | '.join(candidates)}")
                    continue

                try:
                    adj = pd.read_csv(path, header=None).values
                    adj = (adj > 0).astype(int)
                    degrees = adj.sum(axis=1)

                    counts, _ = np.histogram(degrees, bins=bin_edges)
                    # Density-like normalization to keep P(k) scale comparable when BIN_WIDTH changes
                    probs = counts / (N * bin_widths)
                    all_probs.append(probs)

                except Exception as e:
                    print(f"Error reading {path}: {e}")
                    continue

            if all_probs:
                probs_array = np.array(all_probs)
                mean_probs = probs_array.mean(axis=0)
                std_probs = probs_array.std(axis=0)

                ax.bar(bin_centers, mean_probs, width=bin_widths * 0.95, alpha=alfa_mean,
                       color=color_map[eps], label=label_map[eps],
                       edgecolor='black', linewidth=0.5)

                if mostrar_dispersion:
                    ax.fill_between(bin_centers,
                                    mean_probs - std_probs,
                                    mean_probs + std_probs,
                                    color=color_map[eps], alpha=alfa_std)

        # === Subplot aesthetics ===
        ax.set_title(title, fontsize=18)
        ax.set_xlabel("Node Degree $k$", fontsize=18)
        ax.set_ylabel(r"$P(k)$", fontsize=18)  # <- Updated label
        ax.tick_params(axis='both', labelsize=14)
        ax.grid(True)
        ax.set_axisbelow(True)
        if ylim_y is not None:
            ax.set_ylim(0, ylim_y)
        if xlim_x is not None:
            ax.set_xlim(0, xlim_x)

        # Disable scientific notation on axes
        ax.ticklabel_format(style='plain', axis='y')
        ax.ticklabel_format(style='plain', axis='x')

    # === Global legend ===
    handles = [plt.Line2D([0], [0], color=color_map[eps], lw=5, label=label_map[eps]) for eps in epsilons]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.02),
               ncol=len(epsilons), fontsize=16, frameon=False)

    # === Save figure ===
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    output_path = os.path.join(output_dir, f"fig3_k{k}")
    plt.savefig(output_path + ".pdf", bbox_inches="tight")
    plt.savefig(output_path + ".png", dpi=300, bbox_inches="tight")
    plt.close()

    output_label = output_path.replace(os.sep, "/")
    print(f"Final figure saved as {output_label}.[pdf|png]")


# **Figure 4**. Betweenness Distribution.

In [ ]:
from pathlib import Path
import os
import re

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

# Local matplotlib config
MPL_CONFIG_DIR = Path('.mplconfig_betweenness_fixed_eps050')
MPL_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_CONFIG_DIR.resolve()))

# -------------------------
# Style (LaTeX-first)
# -------------------------
USE_TEX = True
ALLOW_TEX_FALLBACK = False  # set True only if you want automatic non-TeX fallback
EPS_SYMBOL = r"\epsilon"

plt.rc('text', usetex=USE_TEX)
plt.rc('font', family='serif')

try:
    _probe = plt.figure(figsize=(1, 1))
    _probe.text(0.5, 0.5, rf'${EPS_SYMBOL}$')
    _probe.canvas.draw()
    plt.close(_probe)
except Exception as exc:
    plt.close('all')
    if ALLOW_TEX_FALLBACK:
        USE_TEX = False
        plt.rc('text', usetex=False)
        print(f"[warn] TeX unavailable ({exc.__class__.__name__}); using mathtext fallback.")
    else:
        raise RuntimeError(
            'LaTeX rendering is required for this figure, but TeX is unavailable in this environment. '
            'Install/configure TeX (or set ALLOW_TEX_FALLBACK=True).'
        ) from exc

# -------------------------
# Config
# -------------------------
K_TO_P = {18: 0.060, 36: 0.120, 54: 0.180}
K_VALUES = [18, 36, 54]
EPS_TARGET = 0.50
FINAL_STEP = 300_000
RESULTS_ROOT = Path('results')
SIMULATION_IDS = None
BETWEENNESS_SAMPLE_K = 80  # keep consistent with exploratory sections

# Moderate/smooth binning for readability
N_BINS = 50

K_COLOR = {18: 'tab:blue', 36: 'tab:orange', 54: 'tab:green'}
K_LABEL = {k: rf'$\langle k \rangle = {k}$' for k in K_VALUES}
ALPHA_FILL = 0.45
BAR_EDGE_LW = 0.45

OUTPUT_DIR = Path("plots_paper/fig4")
OUTPUT_BASENAME = "fig4"


def discover_simulation_ids(base_dir: Path):
    sim_ids = []
    for d in sorted(base_dir.glob('simulation_*')):
        m = re.search(r'simulation_(\d+)$', d.name)
        if m:
            sim_ids.append(int(m.group(1)))
    return sim_ids


def matrix_file(base_dir: Path, sim_id: int, eps: float, step: int):
    eps_tag = f'{eps:.3f}'
    eps_dir = base_dir / f'simulation_{sim_id}' / 'networks' / f'eps_{eps_tag}'
    if not eps_dir.exists():
        return None

    exact = eps_dir / f'matrix_eps_{eps_tag}_step_{step}.txt'
    if exact.exists():
        return exact

    cands = sorted(eps_dir.glob(f'matrix_eps_{eps_tag}_step_{step}*.txt'))
    return cands[0] if cands else None


def load_adjacency_unweighted(path: Path):
    A = np.loadtxt(path, delimiter=',')
    A = np.asarray(A, dtype=float)

    if A.ndim == 1:
        n = int(np.sqrt(A.size))
        if n * n == A.size:
            A = A.reshape(n, n)

    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError(f'Adjacency not square: {path}')

    A = (A > 0).astype(int)
    np.fill_diagonal(A, 0)
    A = np.maximum(A, A.T)
    return A


def compute_node_edge_betweenness(G: nx.Graph, sample_k=None):
    if sample_k is None:
        node_bc = nx.betweenness_centrality(G, normalized=True, weight=None)
        edge_bc = nx.edge_betweenness_centrality(G, normalized=True, weight=None)
    else:
        k_use = min(int(sample_k), max(1, G.number_of_nodes()))
        node_bc = nx.betweenness_centrality(
            G, k=k_use, normalized=True, weight=None, seed=42
        )
        edge_bc = nx.edge_betweenness_centrality(
            G, k=k_use, normalized=True, weight=None, seed=42
        )

    node_vals = np.array(list(node_bc.values()), dtype=float)
    edge_vals = np.array(list(edge_bc.values()), dtype=float)
    return node_vals, edge_vals


def pooled_arrays_for_k(k_val):
    p_val = K_TO_P[k_val]
    base_dir = RESULTS_ROOT / f'results_p{p_val:.3f}'
    if not base_dir.exists():
        return np.array([], dtype=float), np.array([], dtype=float), 0

    sim_ids = discover_simulation_ids(base_dir) if SIMULATION_IDS is None else list(SIMULATION_IDS)

    node_all = []
    edge_all = []
    n_used = 0

    for sim_id in sim_ids:
        f = matrix_file(base_dir, sim_id, EPS_TARGET, FINAL_STEP)
        if f is None:
            continue

        A = load_adjacency_unweighted(f)
        G = nx.from_numpy_array(A)
        node_vals, edge_vals = compute_node_edge_betweenness(G, sample_k=BETWEENNESS_SAMPLE_K)

        node_all.append(node_vals)
        edge_all.append(edge_vals)
        n_used += 1

    if not node_all:
        return np.array([], dtype=float), np.array([], dtype=float), 0

    # Pooling across realizations is ONLY for plotting/visualization.
    return np.concatenate(node_all), np.concatenate(edge_all), n_used


def build_bins(data_dict, percentile=99.5):
    arrs = [v for v in data_dict.values() if v.size > 0]
    if not arrs:
        return None, None
    concat = np.concatenate(arrs)
    x_upper = float(np.percentile(concat, percentile))
    if not np.isfinite(x_upper) or x_upper <= 0:
        x_upper = float(np.max(concat)) if concat.size else 1.0
    x_upper = max(x_upper, 1e-6)
    bins = np.linspace(0.0, x_upper, N_BINS)
    return bins, x_upper


def peak_density(data_dict, bins, x_upper):
    y_max = 0.0
    for vals in data_dict.values():
        if vals.size == 0:
            continue
        vals_clip = np.clip(vals, 0.0, x_upper)
        counts, _ = np.histogram(vals_clip, bins=bins)
        probs = counts / vals_clip.size
        if probs.size:
            y_max = max(y_max, float(np.max(probs)))
    return y_max


# -------------------------
# Load pooled final-time data
# -------------------------
node_by_k = {}
edge_by_k = {}
count_by_k = {}

for k_val in K_VALUES:
    node_vals, edge_vals, n_used = pooled_arrays_for_k(k_val)
    node_by_k[k_val] = node_vals
    edge_by_k[k_val] = edge_vals
    count_by_k[k_val] = n_used


if not any(node_by_k[k].size > 0 for k in K_VALUES):
    raise RuntimeError('No valid data found for final-time fixed-epsilon betweenness figure.')


# -------------------------
# Bins and shared scales
# -------------------------
node_bins, node_x_upper = build_bins(node_by_k)
edge_bins, edge_x_upper = build_bins(edge_by_k)
if node_bins is None or edge_bins is None:
    raise RuntimeError('Could not build histogram bins (insufficient pooled data).')

node_y_max = peak_density(node_by_k, node_bins, node_x_upper)
edge_y_max = peak_density(edge_by_k, edge_bins, edge_x_upper)


# -------------------------
# Plot compact 1x2 figure
# -------------------------
fig, axs = plt.subplots(1, 2, figsize=(14.2, 5.3))

# (a) Node betweenness
ax = axs[0]
for k_val in K_VALUES:
    vals = node_by_k[k_val]
    if vals.size == 0:
        continue
    vals_clip = np.clip(vals, 0.0, node_x_upper)
    ax.hist(
        vals_clip,
        bins=node_bins,
        density=False,
        weights=np.ones_like(vals_clip, dtype=float) / vals_clip.size,
        color=K_COLOR[k_val],
        alpha=ALPHA_FILL,
        edgecolor='black',
        linewidth=BAR_EDGE_LW,
        label=K_LABEL[k_val],
    )
ax.text(0.03, 0.95, '(a)', transform=ax.transAxes, va='top', ha='left', fontsize=13, fontweight='bold')
ax.set_title('Node betweenness', fontsize=16)
ax.set_xlabel(r'$b_n$', fontsize=14)
ax.set_ylabel(r'$P(b_n)$', fontsize=14)
ax.set_xlim(0.0, node_x_upper)
if node_y_max > 0:
    ax.set_ylim(0.0, node_y_max * 1.08)
ax.tick_params(axis='both', labelsize=12)
ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='plain', axis='both')
ax.grid(True, alpha=0.30)
ax.set_axisbelow(True)

# (b) Edge betweenness
ax = axs[1]
for k_val in K_VALUES:
    vals = edge_by_k[k_val]
    if vals.size == 0:
        continue
    vals_clip = np.clip(vals, 0.0, edge_x_upper)
    ax.hist(
        vals_clip,
        bins=edge_bins,
        density=False,
        weights=np.ones_like(vals_clip, dtype=float) / vals_clip.size,
        color=K_COLOR[k_val],
        alpha=ALPHA_FILL,
        edgecolor='black',
        linewidth=BAR_EDGE_LW,
        label=K_LABEL[k_val],
    )
ax.text(0.03, 0.95, '(b)', transform=ax.transAxes, va='top', ha='left', fontsize=13, fontweight='bold')
ax.set_title('Edge betweenness', fontsize=16)
ax.set_xlabel(r'$b_e$', fontsize=14)
ax.set_ylabel(r'$P(b_e)$', fontsize=14)
ax.set_xlim(0.0, edge_x_upper)
if edge_y_max > 0:
    ax.set_ylim(0.0, edge_y_max * 1.08)
ax.tick_params(axis='both', labelsize=12)
ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(style='plain', axis='both')
ax.grid(True, alpha=0.30)
ax.set_axisbelow(True)

handles = [
    plt.Line2D([0], [0], color=K_COLOR[k], lw=4, label=K_LABEL[k], alpha=ALPHA_FILL)
    for k in K_VALUES
]
fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.04),
           ncol=3, fontsize=12, frameon=False)

plt.tight_layout(rect=[0.02, 0.02, 1, 0.95])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / f"{OUTPUT_BASENAME}.pdf", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / f"{OUTPUT_BASENAME}.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("Final figure saved as plots_paper/fig4/fig4.[pdf|png]")


# **Figure 5, 6 and 7**. Community Structure by Louvain Algorithm.

## Figure 5

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import community.community_louvain as community_louvain
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
})

SIMULATION_ID = globals().get("SIMULATION_ID", 1)
FIG5_SIMULATION_ID = globals().get("FIG5_SIMULATION_ID", SIMULATION_ID)
FIG5_K = globals().get("FIG5_K", 18)
FIG5_K_LIST = globals().get("FIG5_K_LIST", [18, 36, 54])
ROOT = Path("results")
out_dir = Path("plots_paper/fig5")
out_dir.mkdir(parents=True, exist_ok=True)

k_to_p = {18: 0.060, 36: 0.120, 54: 0.180}
STATE_CMAP = cm.autumn
STATE_NORM = plt.Normalize(vmin=-1, vmax=1)
LOUVAIN_SEED = 42
LAYOUT_SEED = 42

PANEL_TITLE_FS = 18
PANEL_LABEL_FS = 26
SUPTITLE_FS = 30
EPS_LABEL_FS = 38


def _load_txt(path: Path):
    try:
        return np.loadtxt(path, delimiter=",")
    except Exception:
        return np.loadtxt(path)


def _sci_tex(n: int) -> str:
    if n == 0:
        return "0"
    s = f"{n:.0e}"
    a, b = s.split("e")
    mant = int(float(a))
    exp = int(b)
    return rf"{mant} \times 10^{{{exp}}}"


def _matrix_title_with_Q(step: int, Q: float) -> str:
    q_text = f"{Q:.3f}" if np.isfinite(Q) else "nan"
    return rf"Adaptive time step $= {_sci_tex(step)}$" + "\n" + rf"$Q = {q_text}$"


def _load_ordered_matrix_state(k: int, eps: float, step: int, sim_id: int):
    p_str = f"{k_to_p[k]:.3f}"
    eps_str = f"{eps:.3f}"
    base = ROOT / f"results_p{p_str}" / f"simulation_{sim_id}"
    m_path = base / "networks" / f"eps_{eps_str}" / f"matrix_eps_{eps_str}_step_{step}.txt"
    s_path = base / "states" / f"eps_{eps_str}" / f"states_eps_{eps_str}_step_{step}.txt"

    if (not m_path.exists()) or (not s_path.exists()):
        raise FileNotFoundError(f"Missing files for k={k}, eps={eps_str}, step={step}, sim={sim_id}")

    A = _load_txt(m_path)
    x = np.asarray(_load_txt(s_path)).reshape(-1)

    if A.ndim == 1:
        n = int(np.sqrt(A.size))
        if n * n == A.size:
            A = A.reshape(n, n)

    if x.size != A.shape[0]:
        raise ValueError(f"Size mismatch at step={step}: A has {A.shape[0]}, states has {x.size}")

    G = nx.from_numpy_array(A)
    part = community_louvain.best_partition(G, random_state=LOUVAIN_SEED)
    Q = community_louvain.modularity(part, G) if G.number_of_edges() > 0 else np.nan
    order = np.array(sorted(part.keys(), key=lambda i: (part[i], x[i])), dtype=int)

    A_ord = A[np.ix_(order, order)]
    x_ord = x[order]
    return A_ord, x_ord, G, x, Q


def _draw_matrix_panel(fig, rect, A_ord, x_ord, title, panel_label):
    ax = fig.add_axes(rect)
    vmax = max(1.0, float(np.nanmax(A_ord)))
    ax.imshow(A_ord, cmap="Blues", interpolation="nearest", vmin=0, vmax=vmax, aspect="equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=PANEL_TITLE_FS, pad=10)
    ax.text(1.06, 1.01, rf"$({panel_label})$", transform=ax.transAxes, fontsize=PANEL_LABEL_FS, va="bottom")

    ax_top = ax.inset_axes([0.0, 1.02, 1.0, 0.05], transform=ax.transAxes)
    ax_top.imshow(x_ord.reshape(1, -1), cmap=STATE_CMAP, aspect="auto", norm=STATE_NORM)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_frame_on(False)
    ax_top.set_clip_on(False)

    ax_right = ax.inset_axes([1.02, 0.0, 0.06, 1.0], transform=ax.transAxes)
    ax_right.imshow(x_ord.reshape(-1, 1), cmap=STATE_CMAP, aspect="auto", norm=STATE_NORM)
    ax_right.set_xticks([])
    ax_right.set_yticks([])
    ax_right.set_frame_on(False)
    ax_right.set_clip_on(False)
    return ax


def _draw_graph_panel(fig, rect, G, x, title, panel_label):
    ax = fig.add_axes(rect)
    pos = nx.spring_layout(G, seed=LAYOUT_SEED)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.17, width=0.35, edge_color="gray")
    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=24,
        node_color=x,
        cmap=STATE_CMAP,
        vmin=STATE_NORM.vmin,
        vmax=STATE_NORM.vmax,
        linewidths=0.0,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=PANEL_TITLE_FS, pad=10)
    ax.text(1.04, 1.01, rf"$({panel_label})$", transform=ax.transAxes, fontsize=PANEL_LABEL_FS, va="bottom")
    for spine in ax.spines.values():
        spine.set_visible(False)
    return ax


def _add_arrow(fig, start, end, color, rad=0.0, scale=1.0, z=6):
    arr = FancyArrowPatch(
        start,
        end,
        transform=fig.transFigure,
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle=f"Simple,head_length={14 * scale},head_width={12 * scale},tail_width={3.4 * scale}",
        color=color,
        linewidth=0,
        zorder=z,
    )
    fig.add_artist(arr)


def plot_figure5_story(k: int = FIG5_K, sim_id: int = FIG5_SIMULATION_ID):
    eps_top = 0.30
    eps_bottom = 0.50
    steps = (0, 10000, 100000, 300000)

    A0, x0, _, _, Q0 = _load_ordered_matrix_state(k, eps_top, steps[0], sim_id)

    top_data = [_load_ordered_matrix_state(k, eps_top, s, sim_id) for s in steps[1:]]
    bot_data = [_load_ordered_matrix_state(k, eps_bottom, s, sim_id) for s in steps[1:]]

    _, _, G_top_final, x_top_final, _ = _load_ordered_matrix_state(k, eps_top, steps[-1], sim_id)
    _, _, G_bot_final, x_bot_final, _ = _load_ordered_matrix_state(k, eps_bottom, steps[-1], sim_id)

    fig = plt.figure(figsize=(26, 14), facecolor="white")

    rect_a = [0.055, 0.35, 0.13, 0.25]
    rect_top = [[0.215, 0.63, 0.13, 0.25], [0.41, 0.63, 0.13, 0.25], [0.605, 0.63, 0.13, 0.25]]
    rect_bot = [[0.215, 0.11, 0.13, 0.25], [0.41, 0.11, 0.13, 0.25], [0.605, 0.11, 0.13, 0.25]]
    rect_g_top = [0.80, 0.63, 0.16, 0.25]
    rect_g_bot = [0.80, 0.11, 0.16, 0.25]

    _draw_matrix_panel(
        fig,
        rect_a,
        A0,
        x0,
        _matrix_title_with_Q(steps[0], Q0),
        "a",
    )

    top_labels = ["b", "c", "d"]
    for j, (A_ord, x_ord, _, _, Q) in enumerate(top_data):
        step = steps[j + 1]
        _draw_matrix_panel(
            fig,
            rect_top[j],
            A_ord,
            x_ord,
            _matrix_title_with_Q(step, Q),
            top_labels[j],
        )

    bot_labels = ["f", "g", "h"]
    for j, (A_ord, x_ord, _, _, Q) in enumerate(bot_data):
        step = steps[j + 1]
        _draw_matrix_panel(
            fig,
            rect_bot[j],
            A_ord,
            x_ord,
            _matrix_title_with_Q(step, Q),
            bot_labels[j],
        )

    _draw_graph_panel(
        fig,
        rect_g_top,
        G_top_final,
        x_top_final,
        f"Network graph\nAdaptive time step $= {_sci_tex(steps[-1])}$",
        "e",
    )
    _draw_graph_panel(
        fig,
        rect_g_bot,
        G_bot_final,
        x_bot_final,
        f"Network graph\nAdaptive time step $= {_sci_tex(steps[-1])}$",
        "i",
    )

    fig.text(0.072, 0.905, rf"$\varepsilon = {eps_top:.2f}$", color="blue", fontsize=EPS_LABEL_FS)
    fig.text(0.072, 0.025, rf"$\varepsilon = {eps_bottom:.2f}$", color="red", fontsize=EPS_LABEL_FS)

    _add_arrow(fig, (0.13, 0.66), (0.205, 0.77), color="blue", rad=-0.30, scale=1.1)
    _add_arrow(fig, (0.13, 0.34), (0.205, 0.23), color="red", rad=0.30, scale=1.1)

    _add_arrow(fig, (0.365, 0.755), (0.405, 0.755), color="blue", scale=1.0)
    _add_arrow(fig, (0.56, 0.755), (0.600, 0.755), color="blue", scale=1.0)
    _add_arrow(fig, (0.755, 0.755), (0.795, 0.755), color="blue", scale=1.0)

    _add_arrow(fig, (0.365, 0.235), (0.405, 0.235), color="red", scale=1.0)
    _add_arrow(fig, (0.56, 0.235), (0.600, 0.235), color="red", scale=1.0)
    _add_arrow(fig, (0.755, 0.235), (0.795, 0.235), color="red", scale=1.0)

    _add_arrow(fig, (0.22, 0.495), (0.74, 0.495), color="black", scale=1.7, z=5)
    fig.text(0.445, 0.465, rf"Adaptive time steps for $\langle k \rangle = {k}$", fontsize=26)

    cax = fig.add_axes([0.29, 0.040, 0.43, 0.028])
    sm = cm.ScalarMappable(cmap=STATE_CMAP, norm=STATE_NORM)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label(r"$\mathrm{Dynamic\ range}$", fontsize=30, labelpad=14)
    cbar.ax.tick_params(labelsize=18)

    p_str = f"{k_to_p[k]:.3f}"
    out_base = out_dir / f"fig5_k{k}_p{p_str}_sim_{sim_id}"
    png_file = f"{out_base}.png"
    pdf_file = f"{out_base}.pdf"
    plt.savefig(pdf_file, bbox_inches="tight")
    plt.savefig(png_file, dpi=300, bbox_inches="tight")
    plt.close(fig)
    output_label = str(out_base).replace(os.sep, "/")
    print(f"Final figure saved as {output_label}.[pdf|png]")


for k in FIG5_K_LIST:
    if k not in k_to_p:
        print(f"Skipping unsupported k={k}")
        continue
    plot_figure5_story(k=k, sim_id=FIG5_SIMULATION_ID)

## Figure 6

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import community.community_louvain as community_louvain
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

SIMULATION_ID = globals().get("SIMULATION_ID", 1)
FIG6_SIMULATION_ID = globals().get("FIG6_SIMULATION_ID", SIMULATION_ID)
FIG6_K = globals().get("FIG6_K", 36)
FIG6_EPS = globals().get("FIG6_EPS", 0.50)
ROOT = Path("results")
out_dir = Path("plots_paper/fig6")
out_dir.mkdir(parents=True, exist_ok=True)

k_to_p = {18: 0.060, 36: 0.120, 54: 0.180}
STATE_CMAP = cm.autumn
STATE_NORM = plt.Normalize(vmin=-1, vmax=1)
LOUVAIN_SEED = 42
LAYOUT_SEED = 42

PANEL_TITLE_FS = 26
PANEL_LABEL_FS = 26
GRAPH_TITLE_FS = 27


def _load_txt(path: Path):
    try:
        return np.loadtxt(path, delimiter=",")
    except Exception:
        return np.loadtxt(path)


def _sci_tex(n: int) -> str:
    if n == 0:
        return "0"
    s = f"{n:.0e}"
    a, b = s.split("e")
    mant = int(float(a))
    exp = int(b)
    return rf"{mant} \times 10^{{{exp}}}"


def _matrix_title_with_Q(step: int, Q: float) -> str:
    q_text = f"{Q:.3f}" if np.isfinite(Q) else "nan"
    return rf"$\mbox{{Adaptive\hspace{{0.3em}}time\hspace{{0.3em}}step}} = {_sci_tex(step)}$" + "\n" + rf"$Q = {q_text}$"


def _load_ordered_matrix_state(k: int, eps: float, step: int, sim_id: int):
    p_str = f"{k_to_p[k]:.3f}"
    eps_str = f"{eps:.3f}"
    base = ROOT / f"results_p{p_str}" / f"simulation_{sim_id}"
    m_path = base / "networks" / f"eps_{eps_str}" / f"matrix_eps_{eps_str}_step_{step}.txt"
    s_path = base / "states" / f"eps_{eps_str}" / f"states_eps_{eps_str}_step_{step}.txt"

    if (not m_path.exists()) or (not s_path.exists()):
        raise FileNotFoundError(f"Missing files for k={k}, eps={eps_str}, step={step}, sim={sim_id}")

    A = _load_txt(m_path)
    x = np.asarray(_load_txt(s_path)).reshape(-1)

    if A.ndim == 1:
        n = int(np.sqrt(A.size))
        if n * n == A.size:
            A = A.reshape(n, n)

    if x.size != A.shape[0]:
        raise ValueError(f"Size mismatch at step={step}: A has {A.shape[0]}, states has {x.size}")

    G = nx.from_numpy_array(A)
    part = community_louvain.best_partition(G, random_state=LOUVAIN_SEED)
    Q = community_louvain.modularity(part, G) if G.number_of_edges() > 0 else np.nan
    order = np.array(sorted(part.keys(), key=lambda i: (part[i], x[i])), dtype=int)

    A_ord = A[np.ix_(order, order)]
    x_ord = x[order]
    return A_ord, x_ord, G, x, Q


def _draw_matrix_panel(fig, rect, A_ord, x_ord, title, panel_label):
    ax = fig.add_axes(rect)
    vmax = max(1.0, float(np.nanmax(A_ord)))
    ax.imshow(A_ord, cmap="Blues", interpolation="nearest", vmin=0, vmax=vmax, aspect="equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=PANEL_TITLE_FS, pad=8)
    ax.text(1.12, 1.01, rf"$({panel_label})$", transform=ax.transAxes, fontsize=PANEL_LABEL_FS, va="bottom")

    ax_top = ax.inset_axes([0.0, 1.02, 1.0, 0.045], transform=ax.transAxes)
    ax_top.imshow(x_ord.reshape(1, -1), cmap=STATE_CMAP, aspect="auto", norm=STATE_NORM)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_frame_on(False)
    ax_top.set_clip_on(False)

    ax_right = ax.inset_axes([1.02, 0.0, 0.06, 1.0], transform=ax.transAxes)
    ax_right.imshow(x_ord.reshape(-1, 1), cmap=STATE_CMAP, aspect="auto", norm=STATE_NORM)
    ax_right.set_xticks([])
    ax_right.set_yticks([])
    ax_right.set_frame_on(False)
    ax_right.set_clip_on(False)
    return ax


def _draw_graph_panel(fig, rect, G, x, title, panel_label):
    ax = fig.add_axes(rect)
    pos = nx.spring_layout(G, seed=LAYOUT_SEED)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.17, width=0.35, edge_color="gray")
    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=52,
        node_color=x,
        cmap=STATE_CMAP,
        vmin=STATE_NORM.vmin,
        vmax=STATE_NORM.vmax,
        linewidths=0.0,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=GRAPH_TITLE_FS, pad=8)
    ax.text(1.03, 1.01, rf"$({panel_label})$", transform=ax.transAxes, fontsize=PANEL_LABEL_FS, va="bottom")
    for spine in ax.spines.values():
        spine.set_visible(False)
    return ax


def _add_arrow(fig, start, end, color, rad=0.0, scale=1.0, z=6):
    arr = FancyArrowPatch(
        start,
        end,
        transform=fig.transFigure,
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle=f"Simple,head_length={14 * scale},head_width={12 * scale},tail_width={3.4 * scale}",
        color=color,
        linewidth=0,
        zorder=z,
    )
    fig.add_artist(arr)


def plot_figure6_linear(k: int = FIG6_K, eps: float = FIG6_EPS, sim_id: int = FIG6_SIMULATION_ID):
    if k not in k_to_p:
        raise ValueError(f"Unsupported k={k}. Expected one of {sorted(k_to_p)}")

    steps = (0, 10000, 100000, 300000)
    data = [_load_ordered_matrix_state(k, eps, s, sim_id) for s in steps]
    _, _, G_final, x_final, _ = data[-1]

    fig = plt.figure(figsize=(29, 9), facecolor="white")

    rect_mats = [
        [0.03, 0.22, 0.12, 0.56],
        [0.21, 0.22, 0.12, 0.56],
        [0.39, 0.22, 0.12, 0.56],
        [0.57, 0.22, 0.12, 0.56],
    ]
    rect_graph = [0.74, 0.22, 0.21, 0.56]

    labels = ["a", "b", "c", "d"]
    for i, (A_ord, x_ord, _, _, Q) in enumerate(data):
        _draw_matrix_panel(fig, rect_mats[i], A_ord, x_ord, _matrix_title_with_Q(steps[i], Q), labels[i])

    _draw_graph_panel(
        fig,
        rect_graph,
        G_final,
        x_final,
        f"Network graph\nAdaptive time step $= {_sci_tex(steps[-1])}$",
        "e",
    )

    # Top time arrow and caption
    _add_arrow(fig, (0.14, 0.92), (0.73, 0.92), color="black", scale=2.0, z=5)
    fig.text(0.33, 0.845, rf"Adaptive time steps for $\langle k \rangle = {k}$", fontsize=40)

    # Red transition arrows between panels
    y_arrow = 0.51
    _add_arrow(fig, (0.168, y_arrow), (0.198, y_arrow), color="red", scale=1.1)
    _add_arrow(fig, (0.348, y_arrow), (0.378, y_arrow), color="red", scale=1.1)
    _add_arrow(fig, (0.528, y_arrow), (0.558, y_arrow), color="red", scale=1.1)
    _add_arrow(fig, (0.708, y_arrow), (0.738, y_arrow), color="red", scale=1.1)

    # Dynamic range colorbar
    cax = fig.add_axes([0.24, 0.10, 0.42, 0.035])
    sm = cm.ScalarMappable(cmap=STATE_CMAP, norm=STATE_NORM)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label(r"$\mathrm{Dynamic\ range}$", fontsize=38, labelpad=20)
    cbar.ax.tick_params(labelsize=22)
    fig.text(0.08, 0.115, rf"$\varepsilon = {eps:.2f}$", color="red", fontsize=34)

    p_str = f"{k_to_p[k]:.3f}"
    eps_tag = f"{eps:.3f}".rstrip("0").rstrip(".").replace(".", "_")
    out_base = out_dir / f"fig6_k{k}_p{p_str}_eps_{eps_tag}_sim_{sim_id}"
    png_file = f"{out_base}.png"
    pdf_file = f"{out_base}.pdf"
    plt.savefig(pdf_file, bbox_inches="tight")
    plt.savefig(png_file, dpi=300, bbox_inches="tight")
    plt.close(fig)
    output_label = str(out_base).replace(os.sep, "/")
    print(f"Final figure saved as {output_label}.[pdf|png]")


plot_figure6_linear(k=FIG6_K, eps=FIG6_EPS, sim_id=FIG6_SIMULATION_ID)


## Figure 7

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import community.community_louvain as community_louvain
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

SIMULATION_ID = globals().get("SIMULATION_ID", 1)
FIG7_SIMULATION_ID = globals().get("FIG7_SIMULATION_ID", SIMULATION_ID)
FIG7_K = globals().get("FIG7_K", 54)
FIG7_EPS = globals().get("FIG7_EPS", 0.50)
ROOT = Path("results")
out_dir = Path("plots_paper/fig7")
out_dir.mkdir(parents=True, exist_ok=True)

k_to_p = {18: 0.060, 36: 0.120, 54: 0.180}
STATE_CMAP = cm.autumn
STATE_NORM = plt.Normalize(vmin=-1, vmax=1)
LOUVAIN_SEED = 42
LAYOUT_SEED = 42

PANEL_TITLE_FS = 26
PANEL_LABEL_FS = 26
GRAPH_TITLE_FS = 27


def _load_txt(path: Path):
    try:
        return np.loadtxt(path, delimiter=",")
    except Exception:
        return np.loadtxt(path)


def _sci_tex(n: int) -> str:
    if n == 0:
        return "0"
    s = f"{n:.0e}"
    a, b = s.split("e")
    mant = int(float(a))
    exp = int(b)
    return rf"{mant} \times 10^{{{exp}}}"


def _matrix_title_with_Q(step: int, Q: float) -> str:
    q_text = f"{Q:.3f}" if np.isfinite(Q) else "nan"
    return rf"$\mbox{{Adaptive\hspace{{0.3em}}time\hspace{{0.3em}}step}} = {_sci_tex(step)}$" + "\n" + rf"$Q = {q_text}$"


def _load_ordered_matrix_state(k: int, eps: float, step: int, sim_id: int):
    p_str = f"{k_to_p[k]:.3f}"
    eps_str = f"{eps:.3f}"
    base = ROOT / f"results_p{p_str}" / f"simulation_{sim_id}"
    m_path = base / "networks" / f"eps_{eps_str}" / f"matrix_eps_{eps_str}_step_{step}.txt"
    s_path = base / "states" / f"eps_{eps_str}" / f"states_eps_{eps_str}_step_{step}.txt"

    if (not m_path.exists()) or (not s_path.exists()):
        raise FileNotFoundError(f"Missing files for k={k}, eps={eps_str}, step={step}, sim={sim_id}")

    A = _load_txt(m_path)
    x = np.asarray(_load_txt(s_path)).reshape(-1)

    if A.ndim == 1:
        n = int(np.sqrt(A.size))
        if n * n == A.size:
            A = A.reshape(n, n)

    if x.size != A.shape[0]:
        raise ValueError(f"Size mismatch at step={step}: A has {A.shape[0]}, states has {x.size}")

    G = nx.from_numpy_array(A)
    part = community_louvain.best_partition(G, random_state=LOUVAIN_SEED)
    Q = community_louvain.modularity(part, G) if G.number_of_edges() > 0 else np.nan
    order = np.array(sorted(part.keys(), key=lambda i: (part[i], x[i])), dtype=int)

    A_ord = A[np.ix_(order, order)]
    x_ord = x[order]
    return A_ord, x_ord, G, x, Q


def _draw_matrix_panel(fig, rect, A_ord, x_ord, title, panel_label):
    ax = fig.add_axes(rect)
    vmax = max(1.0, float(np.nanmax(A_ord)))
    ax.imshow(A_ord, cmap="Blues", interpolation="nearest", vmin=0, vmax=vmax, aspect="equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=PANEL_TITLE_FS, pad=8)
    ax.text(1.12, 1.01, rf"$({panel_label})$", transform=ax.transAxes, fontsize=PANEL_LABEL_FS, va="bottom")

    ax_top = ax.inset_axes([0.0, 1.02, 1.0, 0.045], transform=ax.transAxes)
    ax_top.imshow(x_ord.reshape(1, -1), cmap=STATE_CMAP, aspect="auto", norm=STATE_NORM)
    ax_top.set_xticks([])
    ax_top.set_yticks([])
    ax_top.set_frame_on(False)
    ax_top.set_clip_on(False)

    ax_right = ax.inset_axes([1.02, 0.0, 0.06, 1.0], transform=ax.transAxes)
    ax_right.imshow(x_ord.reshape(-1, 1), cmap=STATE_CMAP, aspect="auto", norm=STATE_NORM)
    ax_right.set_xticks([])
    ax_right.set_yticks([])
    ax_right.set_frame_on(False)
    ax_right.set_clip_on(False)
    return ax


def _draw_graph_panel(fig, rect, G, x, title, panel_label):
    ax = fig.add_axes(rect)
    pos = nx.spring_layout(G, seed=LAYOUT_SEED)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.17, width=0.35, edge_color="gray")
    nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax,
        node_size=52,
        node_color=x,
        cmap=STATE_CMAP,
        vmin=STATE_NORM.vmin,
        vmax=STATE_NORM.vmax,
        linewidths=0.0,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=GRAPH_TITLE_FS, pad=8)
    ax.text(1.03, 1.01, rf"$({panel_label})$", transform=ax.transAxes, fontsize=PANEL_LABEL_FS, va="bottom")
    for spine in ax.spines.values():
        spine.set_visible(False)
    return ax


def _add_arrow(fig, start, end, color, rad=0.0, scale=1.0, z=6):
    arr = FancyArrowPatch(
        start,
        end,
        transform=fig.transFigure,
        connectionstyle=f"arc3,rad={rad}",
        arrowstyle=f"Simple,head_length={14 * scale},head_width={12 * scale},tail_width={3.4 * scale}",
        color=color,
        linewidth=0,
        zorder=z,
    )
    fig.add_artist(arr)


def plot_figure7_linear(k: int = FIG7_K, eps: float = FIG7_EPS, sim_id: int = FIG7_SIMULATION_ID):
    if k not in k_to_p:
        raise ValueError(f"Unsupported k={k}. Expected one of {sorted(k_to_p)}")

    steps = (0, 10000, 100000, 300000)
    data = [_load_ordered_matrix_state(k, eps, s, sim_id) for s in steps]
    _, _, G_final, x_final, _ = data[-1]

    fig = plt.figure(figsize=(29, 9), facecolor="white")

    rect_mats = [
        [0.03, 0.22, 0.12, 0.56],
        [0.21, 0.22, 0.12, 0.56],
        [0.39, 0.22, 0.12, 0.56],
        [0.57, 0.22, 0.12, 0.56],
    ]
    rect_graph = [0.74, 0.22, 0.21, 0.56]

    labels = ["a", "b", "c", "d"]
    for i, (A_ord, x_ord, _, _, Q) in enumerate(data):
        _draw_matrix_panel(fig, rect_mats[i], A_ord, x_ord, _matrix_title_with_Q(steps[i], Q), labels[i])

    _draw_graph_panel(
        fig,
        rect_graph,
        G_final,
        x_final,
        f"Network graph\nAdaptive time step $= {_sci_tex(steps[-1])}$",
        "e",
    )

    # Top time arrow and caption
    _add_arrow(fig, (0.14, 0.92), (0.73, 0.92), color="black", scale=2.0, z=5)
    fig.text(0.33, 0.845, rf"Adaptive time steps for $\langle k \rangle = {k}$", fontsize=40)

    # Red transition arrows between panels
    y_arrow = 0.51
    _add_arrow(fig, (0.168, y_arrow), (0.198, y_arrow), color="red", scale=1.1)
    _add_arrow(fig, (0.348, y_arrow), (0.378, y_arrow), color="red", scale=1.1)
    _add_arrow(fig, (0.528, y_arrow), (0.558, y_arrow), color="red", scale=1.1)
    _add_arrow(fig, (0.708, y_arrow), (0.738, y_arrow), color="red", scale=1.1)

    # Dynamic range colorbar
    cax = fig.add_axes([0.24, 0.10, 0.42, 0.035])
    sm = cm.ScalarMappable(cmap=STATE_CMAP, norm=STATE_NORM)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label(r"$\mathrm{Dynamic\ range}$", fontsize=38, labelpad=20)
    cbar.ax.tick_params(labelsize=22)
    fig.text(0.08, 0.115, rf"$\varepsilon = {eps:.2f}$", color="red", fontsize=34)

    p_str = f"{k_to_p[k]:.3f}"
    eps_tag = f"{eps:.3f}".rstrip("0").rstrip(".").replace(".", "_")
    out_base = out_dir / f"fig7_k{k}_p{p_str}_eps_{eps_tag}_sim_{sim_id}"
    png_file = f"{out_base}.png"
    pdf_file = f"{out_base}.pdf"
    plt.savefig(pdf_file, bbox_inches="tight")
    plt.savefig(png_file, dpi=300, bbox_inches="tight")
    plt.close(fig)
    output_label = str(out_base).replace(os.sep, "/")
    print(f"Final figure saved as {output_label}.[pdf|png]")


plot_figure7_linear(k=FIG7_K, eps=FIG7_EPS, sim_id=FIG7_SIMULATION_ID)
